# SIH26053 Phase 1 — Reproducible Kaggle Environment

This notebook validates the development environment and the complete `v1.0-mini` nuScenes/lidarseg data contract before mapping or model work begins.

Required Kaggle Secrets: `FOVEAMAP_REPO_URL` and `GITHUB_READ_TOKEN`. The repository token should be a fine-grained token restricted to one private repository with Contents read-only access.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import tempfile
from urllib.parse import urlsplit, urlunsplit

IN_KAGGLE = Path('/kaggle/input').exists()
WORKING_DIR = Path('/kaggle/working') if IN_KAGGLE else Path.cwd() / 'artifacts'
ARTIFACT_DIR = WORKING_DIR / 'artifacts' / 'phase1'
REPO_DIR = WORKING_DIR / 'sih-foveamap'
REPO_REF = 'main'
DATASET_SLUG = 'vyomkeshsharma/nuscenes-mini-complete-with-lidarseg'
SEED = 26053
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Kaggle runtime: {IN_KAGGLE}')
print(f'Working directory: {WORKING_DIR}')
print(f'Artifact directory: {ARTIFACT_DIR}')


## Gate 1 — Private repository checkout

The token is passed through a temporary `GIT_ASKPASS` process and is never embedded in the clone URL, saved remote, or notebook output.


In [ ]:
def clone_private_repo(repo_url: str, token: str, destination: Path, ref: str) -> str:
    parts = urlsplit(repo_url.strip())
    if parts.scheme != 'https' or not parts.netloc or not parts.path or '@' in parts.netloc:
        raise ValueError('FOVEAMAP_REPO_URL must be a clean HTTPS Git URL without credentials')
    clean_url = urlunsplit((parts.scheme, parts.netloc, parts.path, '', ''))
    auth_url = urlunsplit((parts.scheme, f'x-access-token@{parts.netloc}', parts.path, '', ''))
    if destination.exists():
        shutil.rmtree(destination)
    with tempfile.TemporaryDirectory(prefix='foveamap-git-') as temp_dir:
        askpass = Path(temp_dir) / 'askpass.sh'
        askpass.write_text('#!/bin/sh\nprintf "%s\\n" "$GITHUB_READ_TOKEN"\n', encoding='utf-8')
        askpass.chmod(0o700)
        env = os.environ.copy()
        env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_READ_TOKEN': token})
        try:
            subprocess.run(['git', 'clone', auth_url, str(destination)], check=True, env=env, capture_output=True, text=True)
            subprocess.run(['git', '-C', str(destination), 'remote', 'set-url', 'origin', clean_url], check=True, env=env, capture_output=True, text=True)
            subprocess.run(['git', '-C', str(destination), 'checkout', ref], check=True, env=env, capture_output=True, text=True)
            commit = subprocess.run(['git', '-C', str(destination), 'rev-parse', 'HEAD'], check=True, env=env, capture_output=True, text=True).stdout.strip()
        except subprocess.CalledProcessError as exc:
            if destination.exists():
                shutil.rmtree(destination)
            stderr = (exc.stderr or 'git checkout failed').replace(token, '***')
            raise RuntimeError(f'Private repository checkout failed: {stderr}') from exc
    return commit

if IN_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    try:
        repo_url = secrets.get_secret('FOVEAMAP_REPO_URL')
        github_token = secrets.get_secret('GITHUB_READ_TOKEN')
    except Exception as exc:
        raise RuntimeError('Attach FOVEAMAP_REPO_URL and GITHUB_READ_TOKEN in Kaggle Secrets, then rerun') from exc
    PROJECT_ROOT = REPO_DIR
    CHECKOUT_SHA = clone_private_repo(repo_url, github_token, PROJECT_ROOT, REPO_REF)
else:
    PROJECT_ROOT = Path.cwd()
    CHECKOUT_SHA = subprocess.run(['git', 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()

SOURCE_DIR = PROJECT_ROOT / 'src'
if str(SOURCE_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR.resolve()))

print(f'Project root: {PROJECT_ROOT}')
print(f'Git commit: {CHECKOUT_SHA}')


## Gate 2 — Minimal dependency bootstrap

Only the pinned nuScenes devkit is installed. Kaggle's existing PyTorch/CUDA stack is preserved.


In [ ]:
requirements = PROJECT_ROOT / 'requirements' / 'kaggle.txt'
if not requirements.exists():
    raise FileNotFoundError(f'Missing requirements file: {requirements}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-r', str(requirements)], check=True)

import foveamap
from foveamap.runtime import collect_runtime_report, write_json
from foveamap.data.discovery import discover_dataset_root
from foveamap.data.audit import audit_nuscenes_mini
from foveamap.data.sample import load_labelled_sample, summarize_labelled_sample
from foveamap.visualization import render_height_bev, render_semantic_bev

print(f'foveamap version: {foveamap.__version__}')


## Gate 3 — Runtime and accelerator diagnostics


In [ ]:
runtime_report = collect_runtime_report(PROJECT_ROOT, seed=SEED)
environment_path = write_json(runtime_report, ARTIFACT_DIR / 'environment.json')
print(f'Accelerator: {runtime_report["accelerator"]["device"]}')
print(f'PyTorch: {runtime_report["packages"].get("torch")}')
print(f'Environment report: {environment_path}')


## Gate 4 — Dataset-root discovery


In [ ]:
input_root = Path('/kaggle/input') if IN_KAGGLE else Path.cwd()
dataset_root = discover_dataset_root(input_root, preferred_slug=DATASET_SLUG)
print(f'Dataset root: {dataset_root.path}')


## Gate 5 — Fast metadata and representative pair check


In [ ]:
fast_audit = audit_nuscenes_mini(dataset_root.path, max_point_pairs=3)
write_json(fast_audit, ARTIFACT_DIR / 'dataset_audit_fast.json')
print(f'Scenes: {fast_audit["actual"]["scenes"]}')
print(f'Samples: {fast_audit["actual"]["samples"]}')
print(f'LIDAR keyframes: {fast_audit["actual"]["lidar_keyframes"]}')
print(f'Mini lidarseg records: {fast_audit["actual"]["lidarseg_records"]}')
print(f'Representative pairs checked: {fast_audit["actual"]["pairs_checked"]}')
if not fast_audit['passed']:
    raise RuntimeError('Fast dataset audit failed; inspect dataset_audit_fast.json')


## Gate 6 — Complete 404-frame point/label audit


In [ ]:
dataset_audit = audit_nuscenes_mini(dataset_root.path, max_point_pairs=None)
dataset_audit_path = write_json(dataset_audit, ARTIFACT_DIR / 'dataset_audit.json')
print(f'Pairs checked: {dataset_audit["actual"]["pairs_checked"]}')
print(f'Ignored trainval label files: {dataset_audit["actual"]["ignored_trainval_label_files"]}')
print(f'Dataset audit: {dataset_audit_path}')
if not dataset_audit['passed']:
    failed = [name for name, passed in dataset_audit['checks'].items() if not passed]
    raise RuntimeError(f'Full dataset audit failed: {failed}')


## Gate 7 — Labelled point-cloud sanity visualization


In [ ]:
sample = load_labelled_sample(dataset_root.path)
sample_summary = summarize_labelled_sample(sample)
write_json(sample_summary, ARTIFACT_DIR / 'sample_summary.json')
height_path = render_height_bev(sample.points, ARTIFACT_DIR / 'sample_height_bev.png')
semantic_path = render_semantic_bev(
    sample.points,
    sample.labels,
    sample.category_names,
    ARTIFACT_DIR / 'sample_semantic_bev.png',
)
print(f'Point count: {sample_summary["point_count"]}')
print(f'Label count: {sample_summary["label_count"]}')
print(f'Height BEV: {height_path}')
print(f'Semantic BEV: {semantic_path}')


## Gate 8 — Final manifest


In [ ]:
phase1_summary = {
    'git_sha': CHECKOUT_SHA,
    'accelerator': runtime_report['accelerator']['device'],
    'dataset': DATASET_SLUG,
    'dataroot': str(dataset_root.path),
    'version': 'v1.0-mini',
    'fast_audit_passed': fast_audit['passed'],
    'full_audit_passed': dataset_audit['passed'],
    'samples': dataset_audit['actual']['samples'],
    'lidarseg_records': dataset_audit['actual']['lidarseg_records'],
    'pairs_checked': dataset_audit['actual']['pairs_checked'],
    'ignored_trainval_label_files': dataset_audit['actual']['ignored_trainval_label_files'],
    'artifacts': sorted(path.name for path in ARTIFACT_DIR.iterdir() if path.is_file()),
}
summary_path = write_json(phase1_summary, ARTIFACT_DIR / 'phase1_summary.json')
print(json.dumps(phase1_summary, indent=2))
print(f'Phase 1 manifest: {summary_path}')
if not phase1_summary['full_audit_passed']:
    raise RuntimeError('Phase 1 did not pass its dataset contract gate')
